# Order Reviews Data Cleaning

Problems to Investigate
- Change creation date and answer timestamp to time objects
- Review answer is after review creation date
- 814 duplicate reviews
    - 1603 rows with duplicate review Ids
    - Are duplicate review Ids associated w multiple orders?
- Missing review comment titles and messages
- All reviews connect to an existing order
    - Also check if reviews are associated with canceled/undelivered orders
- All review ids are unique
- All review scores fall within 1-5 range
- Check for leading/trailing whitespace in columns and values


In [1]:
import pandas as pd 

reviews_raw = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
reviews = reviews_raw.copy()

print("Original Shape:", reviews_raw.shape)
print("Working Shape:)", reviews.shape)

Original Shape: (99224, 7)
Working Shape:) (99224, 7)


In [2]:
# Convert timestamps to datetime
dates = ["review_creation_date", "review_answer_timestamp"]

for field in dates:
    reviews[field] = pd.to_datetime(reviews[field], errors="coerce")


print("Check conversions:\n", reviews[dates].dtypes)
print("\nCheck null vals:\n", reviews[dates].isnull().sum())

Check conversions:
 review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

Check null vals:
 review_creation_date       0
review_answer_timestamp    0
dtype: int64


In [4]:
# Are all review answers submitted after the review creation date

invalid_response = reviews[
    (reviews["review_answer_timestamp"] < reviews["review_creation_date"])]

invalid_response

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [7]:
# Investigate duplicate reviews

duplicate_reviews = reviews[
    reviews["review_id"].duplicated(keep=False)].sort_values("review_id")

duplicate_reviews.head(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44


In [12]:
# Are the review comments for the duplicate review ids the same

review_content = [
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
]

review_content_comparison = (
    duplicate_reviews.groupby("review_id")[review_content]
    .nunique(dropna=False)
)

same_content = review_content_comparison.eq(1).all(axis=1)

print("Duplicate review IDs:",
      duplicate_reviews["review_id"].nunique())

print("Duplicate review rows:",
      len(duplicate_reviews))

print("Same content:",
      same_content.sum())

print("Different content:",
      (~same_content).sum())

Duplicate review IDs: 789
Duplicate review rows: 1603
Same content: 789
Different content: 0


In [15]:
# Investigating orders associated with duplicate reviews
order_count = (
    duplicate_reviews.groupby("review_id")["order_id"]
    .nunique()
)

print("Review IDs associated with multiple orders:",
      (order_count > 1).sum())

print("Maximum orders associated with one review ID:",
      order_count.max())

Review IDs associated with multiple orders: 789
Maximum orders associated with one review ID: 3


In [16]:
duplicate_order_mapping = (
    duplicate_reviews.groupby("review_id")
    .agg(
        order_ids=("order_id", list),
        number_of_orders=("order_id", "nunique")
    )
    .reset_index()
)

duplicate_order_mapping[
    duplicate_order_mapping["number_of_orders"] > 1
].head(20)

,review_id,order_ids,number_of_orders
0,00130cbe1f9d422698c812ed8ded1919,"[dfcdfc43867d1c1381bfaf62d6b9c195, 04a28263e08...",2
1,0115633a9c298b6a98bcbe4eee75345f,"[78a4201f58af3463bdab842eea4bc801, 0c9850b2c17...",2
2,0174caf0ee5964646040cd94e15ac95e,"[f93a732712407c02dce5dd5088d0f47b, 74db91e33b4...",2
3,017808d29fd1f942d97e50184dfb4c13,"[8daaa9e99d60fbba579cc1c3e3bfae01, b1461c88821...",2
4,0254bd905dc677a6078990aad3331a36,"[5bf226cf882c5bf4247f89a97c86f273, 331b367bdd7...",2
5,0288d42bef3dfe36930740c9588a570f,"[33d8795f04dd631f3480d7aaf90da3dc, f889a5a0b44...",2
6,02aa7f5f75e964e3c7efa59a1f515281,"[d8e17d5f7dacf0970d316e7c03e741e8, db92613c074...",2
7,034528cb00b6bc981847acafbdf2ae0b,"[e1e8e3bca903de27e9a1c72b5a5795e0, 3d3742a96f2...",2
8,03a6a25db577d0689440933055111897,"[2acfdc5131ff2cf4433e668454c9784c, 3fde8b7313a...",2
9,0467560f511c516ddaa54a60edb0c291,"[0c995611a99f81268d859184a416f1db, 55b2e390d5d...",2


In [24]:
# Investigate orders associated with duplicate reviews

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders = pd.read_csv(
    "../data/processed/olist_orders_cleaned.csv",
    parse_dates=date_columns
)

order_investigation = duplicate_reviews[
    ["review_id", "order_id"]
].merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="left"
)

print(order_investigation["order_status"].value_counts())

order_status
delivered      1492
canceled         61
shipped          28
unavailable      10
processing        6
invoiced          6
Name: count, dtype: int64


In [26]:
order_investigation = duplicate_reviews[
    ["review_id", "order_id"]
].merge(
    orders[[
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    ]],
    on="order_id",
    how="left"
)

order_investigation.sort_values(
    ["review_id", "order_purchase_timestamp"]
).head(30)

,review_id,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date
0,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,delivered,2018-02-02 18:01:08,2018-02-26 18:18:45
1,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,delivered,2018-02-02 18:01:10,2018-03-11 11:32:24
3,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,delivered,2017-09-04 15:17:11,2017-09-18 19:43:06
2,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,delivered,2017-09-04 15:17:12,2017-09-20 22:24:03
5,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,delivered,2018-02-15 13:09:28,2018-02-24 00:33:11
4,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,delivered,2018-02-15 13:09:29,2018-03-06 21:39:06
6,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,delivered,2018-02-26 17:35:23,2018-03-02 17:35:37
7,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,delivered,2018-02-26 17:35:24,2018-03-01 17:52:00
8,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,delivered,2017-09-04 23:02:26,2017-09-08 19:31:52
9,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,delivered,2017-09-04 23:02:26,2017-09-14 20:22:47


In [28]:
# Time between orders associated with duplicate reviews
purchase_date_summary = (
    order_investigation.groupby("review_id")
    ["order_purchase_timestamp"]
    .agg(["min", "max"])
)

purchase_date_summary["hours_between_orders"] = (
    purchase_date_summary["max"] -
    purchase_date_summary["min"]
).dt.total_seconds() / 3600

purchase_date_summary["hours_between_orders"].describe()

count     789.000000
mean       39.501024
std       215.291858
min         0.000000
25%         0.000000
50%         0.000278
75%         0.000556
max      3051.404167
Name: hours_between_orders, dtype: float64

In [29]:
# Investigate missing review comments and titles

missing_comment_summary = pd.DataFrame({
    "Missing Titles": [reviews["review_comment_title"].isna().sum()],
    "Missing Messages": [reviews["review_comment_message"].isna().sum()],
    "Both Missing": [
        (reviews["review_comment_title"].isna() &
         reviews["review_comment_message"].isna()).sum()
    ],
    "Neither Missing": [
        (reviews["review_comment_title"].notna() &
         reviews["review_comment_message"].notna()).sum()
    ]
})

missing_comment_summary

,Missing Titles,Missing Messages,Both Missing,Neither Missing
0,87656,58247,56518,9839


In [30]:
print("Title present, message missing:",
      (reviews["review_comment_title"].notna() &
       reviews["review_comment_message"].isna()).sum())

print("Message present, title missing:",
      (reviews["review_comment_title"].isna() &
       reviews["review_comment_message"].notna()).sum())

Title present, message missing: 1729
Message present, title missing: 31138


In [31]:
for column in ["review_comment_title", "review_comment_message"]:
    empty_count = (
        reviews[column].notna() &
        reviews[column].str.strip().eq("")
    ).sum()

    print(f"{column}: {empty_count} empty/whitespace-only values")

review_comment_title: 2 empty/whitespace-only values
review_comment_message: 27 empty/whitespace-only values


In [33]:
for column in ["review_comment_title", "review_comment_message"]:
    reviews[column] = reviews[column].replace(
        r"^\s*$", pd.NA, regex=True
    )

for column in ["review_comment_title", "review_comment_message"]:
    empty_count = (
        reviews[column].notna() &
        reviews[column].str.strip().eq("")
    ).sum()

    print(f"{column}: {empty_count} empty/whitespace-only values")

review_comment_title: 0 empty/whitespace-only values
review_comment_message: 0 empty/whitespace-only values


In [34]:
print("Title present, message missing:",
      (reviews["review_comment_title"].notna() &
       reviews["review_comment_message"].isna()).sum())

print("Message present, title missing:",
      (reviews["review_comment_title"].isna() &
       reviews["review_comment_message"].notna()).sum())

Title present, message missing: 1737
Message present, title missing: 31121


In [37]:
# Are all order ids valid/existing
review_order_check = reviews.merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="left",
    indicator=True
)

print(review_order_check["_merge"].value_counts())

_merge
both          99224
left_only         0
right_only        0
Name: count, dtype: int64


In [36]:
# Validate review scores

print(reviews["review_score"].value_counts())

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64


In [38]:
# Clean any trailing whitespace in columns
reviews.columns = (
    reviews.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(reviews.columns.tolist())


['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
